# NOTEARS experiment analysis

In [1]:
from pathlib import Path
import json
import warnings
import re
from prior_notears import utils
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from prior_notears.utils import generate_prior_knowledge

In [3]:
# DAG: 0 → 1 → 2 → 3 → 4
# B[i, j] = 1 means i → j.
B = np.array([
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0],
], dtype=int)

priors = generate_prior_knowledge(
    B,
    prior_rate=0.5,
    prior_type="exist_path_pairs",
    random_state=42,
    imperfect=True,
    imperfect_rate=0.3,
)

print(priors)

{'exist_path_pairs': [(2, 4), (1, 3), (1, 0), (0, 2), (2, 0)]}


## grid search

In [ ]:
INPUT_DIRS = {
    "linear": Path("grid_search"),
}

ACC_KEYS = {
    "acc_prior_ll": ("with_prior", "likelihood"),
    "acc_prior_l2": ("with_prior", "l2"),
    "acc_no_prior_ll": ("no_prior", "likelihood"),
    "acc_no_prior_l2": ("no_prior", "l2"),
}

In [ ]:
def load_accuracy_rows(input_dirs):
    rows = []

    for experiment, directory in input_dirs.items():
        for path in sorted(directory.glob("*.json")):
            with path.open(encoding="utf-8") as file:
                result = json.load(file)

            d = len(result.get("B_true", [])) or None
            seed_match = re.search(r"_seed(\d+)(?=_|\.json$)", path.name)
            seed = int(seed_match.group(1)) if seed_match else None

            epsilon_match = re.search(r"_epsilon([^_]+)(?=_seed)", path.stem)
            epsilon = (
                float(epsilon_match.group(1))
                if epsilon_match else float("nan")
            )

            configuration = re.sub(r"_seed\d+(?=_|$)", "", path.stem)

            for key, (prior_setting, loss) in ACC_KEYS.items():
                scores = result.get(key)
                if not isinstance(scores, dict):
                    continue

                row = {
                    "experiment": experiment,
                    "prior_setting": prior_setting,
                    "loss": loss,
                    "d": d,
                    "seed": seed,
                    "epsilon": epsilon,
                    "configuration": configuration,
                }
                row.update({
                    name: value
                    for name, value in scores.items()
                    if isinstance(value, (int, float))
                })
                rows.append(row)

    return pd.DataFrame(rows)


accuracy_rows = load_accuracy_rows(INPUT_DIRS)
accuracy_rows.head()

In [ ]:
metadata_columns = {
    "experiment", "configuration", "prior_setting",
    "loss", "d", "seed", "epsilon", "file",
}
score_columns = [
    column for column in accuracy_rows.columns
    if column not in metadata_columns
    and pd.api.types.is_numeric_dtype(accuracy_rows[column])
]

# Average runs within each setup; f1_std is sample standard deviation (ddof=1).
summary = (
    accuracy_rows
    .groupby(
        ["experiment", "d", "epsilon", "prior_setting", "loss"],
        dropna=False,
    )
    .agg(num_seeds=("seed", "nunique"), f1_std=("f1", "std"), **{
        score: (score, "mean") for score in score_columns
    })
    .round(4)
)
summary

### Conclusion: epsilon 0.1 is the best

## Exist Paths


In [6]:
# Change these paths if the result directories are elsewhere.
INPUT_DIRS = {
    "lbfgsb soft": Path("linear_exist_path_pairs_lbfgsb_soft"),
    "adam soft": Path("linear_exist_path_pairs_soft"),
    "nash mtl": Path("linear_exist_path_pairs_nash"),
    "CAGrad mtl": Path("linear_exist_path_pairs_cagrad"),
}


ACC_KEYS = {
    "acc_prior_ll": ("with_prior", "likelihood"),
    "acc_prior_l2": ("with_prior", "l2"),
    "acc_no_prior_ll": ("no_prior", "likelihood"),
    "acc_no_prior_l2": ("no_prior", "l2"),
}

In [8]:
def load_accuracy_rows(input_dirs):
    rows = []

    for experiment, directory in input_dirs.items():

        paths = sorted(directory.glob("*.json"))
        if not paths:
            warnings.warn(
                f"No JSON results for {experiment!r} in {directory.resolve()}",
                stacklevel=2,
            )
        for path in paths:
            with path.open(encoding="utf-8") as file:
                result = json.load(file)

            d = len(result.get("B_true", [])) or None
            seed_match = re.search(r"_seed(\d+)(?=_|\.json$)", path.name)
            seed = int(seed_match.group(1)) if seed_match else None
            epsilon_match = re.search(r"_epsilon([^_]+)(?=_seed)", path.stem)
            epsilon = float(epsilon_match.group(1)) if epsilon_match else float("nan")
            configuration = re.sub(r"_seed\d+(?=_|$)", "", path.stem)
            compare = "compare" in path.name
            tau_match = re.search(r"_tau([^_]+)(?=_|$)", path.stem)
            # tau = float(tau_match.group(1)) if tau_match else float("nan")
            for key, (prior_setting, loss) in ACC_KEYS.items():
                scores = result.get(key)

                if not isinstance(scores, dict):
                    continue
                
                row = {
                    "experiment": experiment,
                    "prior_setting": prior_setting,
                    "loss": loss,
                    "d": d,
                    "seed": seed,
                    "epsilon": epsilon,
                    "configuration": configuration,
                    "file": path.name,
                    "compare": compare,
                    # "tau": tau,
                    "satisfied_percentage": result.get(
                    key.replace("acc_", "satisfied_percentage_", 1),
                    float("nan"),
                ),
                }
                row.update({
                    score_name: score_value
                    for score_name, score_value in scores.items()
                    if isinstance(score_value, (int, float))
                })
                rows.append(row)

    return pd.DataFrame(rows)

accuracy_rows = load_accuracy_rows(INPUT_DIRS)

num_files = accuracy_rows[["experiment", "file"]].drop_duplicates().shape[0]
print(f"Loaded {len(accuracy_rows)} accuracy records from {num_files} JSON files.")
print(accuracy_rows.groupby("experiment")["file"].nunique().rename("JSON files"))
accuracy_rows.loc[
    accuracy_rows["loss"].eq("likelihood")
    & accuracy_rows["prior_setting"].eq("with_prior")
    & accuracy_rows["d"].eq(20)
].pivot(index="seed", columns="experiment", values="f1").round(4)

Loaded 767 accuracy records from 200 JSON files.
experiment
CAGrad mtl     50
adam soft      50
lbfgsb soft    50
nash mtl       50
Name: JSON files, dtype: int64


experiment,CAGrad mtl,adam soft,lbfgsb soft,nash mtl
seed,,,,
0,0.6667,0.6000,0.8333,0.4545
1,0.7692,0.7179,0.6667,0.4103
2,0.8649,0.8649,0.8108,0.8421
3,0.6667,0.5789,0.5789,0.6500
4,0.5000,0.4000,NaN,0.4737
5,0.7027,0.8333,0.8333,0.8293
6,0.3784,0.3784,0.4444,0.5366
7,0.5143,0.5143,0.5143,0.4500
8,0.9000,0.9000,0.8293,0.8293


In [ ]:
expected_seeds = set(range(10))
metadata_columns = {
    "experiment", "configuration", "prior_setting",
    "loss", "d", "seed", "epsilon", "file", "compare"
}
score_columns = [
    column for column in accuracy_rows.columns
    if column not in metadata_columns
    and pd.api.types.is_numeric_dtype(accuracy_rows[column])
]
summary = (
    accuracy_rows
    .groupby(
        ["experiment", "d", "prior_setting", "loss", "compare"],
        dropna=False,
    )
    .agg(
        num_seeds=("seed", "nunique"),
        failed_seeds=(
            "seed",
            lambda seeds: sorted(
                expected_seeds - set(seeds.dropna().astype(int))
            ),
        ),
        f1_std=("f1", "std"),
        **{score: (score, "mean") for score in score_columns},
    )
    .round(4)
)
summary.xs(False, level="compare").xs("with_prior", level="prior_setting")

In [ ]:

def plot_f1_summary(summary):
    data = summary.reset_index()

    if "compare" in data.columns:
        data = data[data["compare"].eq(False)]

    # Plot the methods actually loaded by INPUT_DIRS.
    colors = plt.get_cmap("tab10").colors
    markers = ["o", "s", "D", "^", "v", "P", "X"]
    experiments = data["experiment"].drop_duplicates().tolist()
    styles = [
        (experiment, "with_prior", experiment, colors[i % len(colors)],
         markers[i % len(markers)], "-")
        for i, experiment in enumerate(experiments)
    ]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

    for ax, loss in zip(axes, ["likelihood", "l2"]):
        for experiment, setting, label, color, marker, line in styles:
            subset = data[
                data["experiment"].eq(experiment)
                & data["prior_setting"].eq(setting)
                & data["loss"].eq(loss)
            ].sort_values("d")

            if subset.empty:
                continue
            if subset["d"].duplicated().any():
                raise ValueError(f"Multiple rows per d for {label}, {loss}")

            ax.errorbar(
                subset["d"], subset["f1"],
                yerr=subset["f1_std"].fillna(0),
                label=label,
                color=color,
                marker=marker,
                linestyle=line,
                capsize=4,
            )

        ax.set_title("Likelihood" if loss == "likelihood" else "L2")
        ax.set_xlabel("Number of nodes d")
        ax.set_xticks(sorted(data["d"].dropna().unique()))
        ax.grid(alpha=0.3)

    axes[0].set_ylabel("Mean F1")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=max(1, len(handles)))
    fig.tight_layout(rect=(0, 0.12, 1, 1))
    return fig, axes

fig, axes = plot_f1_summary(summary)
plt.show()

# Optional:
# fig.savefig("f1_comparison.png", dpi=300, bbox_inches="tight")